In [ ]:
# load the dataset

GDP_MM_Correlation_df = pd.read_csv('data/GDP_MM_processed.csv')

GDP_MM_Correlation_df.head()

In [ ]:
# Use pd.cut to create the Affordability_Rating column
GDP_MM_Correlation_df['Affordability_Rating'] = pd.cut(
     GDP_MM_Correlation_df['Median_Multiple'],
     bins=[0, 3.0, 4.0, float('inf')],
     labels=['Affordable', 'Moderately_Unaffordable', 'Seriously_Unaffordable'])

# note: this code leaves out higher levels of unaffordability classification 
# because they are not relevent to this analysis but it should be updated for additional/other data

# Display the first few rows to verify
GDP_MM_Correlation_df.head() 

In [ ]:
# Define custom colors for Affordability Rating with correct category names
colors = {
    'Affordable': 'green',
    'Moderately_Unaffordable': 'orange',
    'Seriously_Unaffordable': 'red'}

In [ ]:
# Creation of a random forest classification model

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt


# Prepare features and target
X = GDP_MM_Correlation_df[["real_GDP"]]
y = GDP_MM_Correlation_df["Affordability_Rating"]

# Handle imbalanced classes with SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=42, stratify=y_resampled)

# Create and train model with more focused hyperparameter optimization
param_grid = {
    'n_estimators': [100, 200],  
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

# Use more CPU cores if available
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='balanced_accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Evaluate best model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, digits=4))


# Create confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.xticks(rotation=45)
plt.show()


# Show distribution of classes before and after SMOTE
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
y_counts = GDP_MM_Correlation_df["Affordability_Rating"].value_counts()
plt.bar(y_counts.index, y_counts.values)
plt.title('Class Distribution Before SMOTE')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
y_resampled_counts = pd.Series(y_resampled).value_counts()
plt.bar(y_resampled_counts.index, y_resampled_counts.values)
plt.title('Class Distribution After SMOTE')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Random Forest Classification Results Analysis

Model Performance Summary

The Random Forest classification model achieved excellent results in predicting housing affordability categories based on GDP data. Using a SMOTE function to balance class distribution the model demonstrates strong predictive capabilities across all affordability classes.

Hyperparameter Configuration

The optimal model configuration identified through GridSearchCV includes:
- 100 decision trees in the ensemble
- No maximum depth restriction, allowing trees to grow to their full extent
- Minimum of 2 samples required to split an internal node
- Balanced class weighting to account for class imbalance in the original dataset

Classification Performance by Category

The model demonstrated consistent performance across all affordability categories:


Implications
 
The model can reliably identify timeframe likely to experience housing affordability challenges based solely on economic output data. This suggests reconsidering GDP growth as inherently positive for society and consideration of other metrics to improve societal cohesion.


Next Steps

This work should be reviewed with subject matter experts as I do not have subject matter expertise in this area. I am also not experienced with time series analysis and there may be additional considerations that I could have missed.
Geographic sub-analysis can and should be done to identify regional variations in the GDP-affordability and evaluate consistency.